In [55]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [56]:
# Parameters
IMG_SIZE = 224           # Define the input image size (adjust as needed)
BATCH_SIZE = 32          # Batch size for training
EPOCHS = 10              # Number of epochs (adjust based on dataset size)
NUM_CLASSES = 5          # Update with the number of classes in your dataset

In [57]:
# Data augmentation and preprocessing
# Use validation_split to reserve a part of your dataset for validation
datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2  # 20% of data used for validation
)

In [58]:
# Directory containing your training images arranged in class subdirectories
DATA_DIR = 'C:/Users/5A_Traders/Desktop/datasets/Chess'  # <-- Change this to your dataset path


In [59]:
# Training and validation generators
train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


Found 105 images belonging to 5 classes.
Found 23 images belonging to 5 classes.


In [60]:
NUM_CLASSES = train_generator.num_classes

In [61]:
# Load the pre-trained MobileNetV2 model without its top classifier
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))


In [62]:
# Freeze the base model to retain the pre-trained features during initial training
base_model.trainable = False

In [63]:
# Create the classification head
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

In [64]:
# Use the base model in inference mode (no dropout, etc.)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)  # Regularization layer to help avoid overfitting
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

In [65]:
# Assemble the complete model
model = Model(inputs, outputs)

In [66]:
# Compile the model with an appropriate optimizer and loss function
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [67]:
# Summary of the model architecture
#model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_4           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 5)                   │           6,405 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [68]:
# Train the model
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS
)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step - accuracy: 0.2607 - loss: 1.9818 - val_accuracy: 0.0870 - val_loss: 1.7620
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.2743 - loss: 1.6685 - val_accuracy: 0.0870 - val_loss: 1.5932
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.3566 - loss: 1.5984 - val_accuracy: 0.3043 - val_loss: 1.5162
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.4570 - loss: 1.3077 - val_accuracy: 0.3478 - val_loss: 1.3533
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.5708 - loss: 1.1806 - val_accuracy: 0.3043 - val_loss: 1.4326
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.5291 - loss: 1.2496 - val_accuracy: 0.3913 - val_loss: 1.3734
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 955ms/step - accuracy: 0.6339 - loss: 1.0004 - val_accuracy: 0.4783 - val_loss: 1.1052
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.5942 - loss: 0.9784 - val_accuracy: 0.5217 - val_loss: 1.1956
Epoch